In [1]:
# uncomment the following code to redownload the weather data for Waterloo, Ontario for the year 2025. 
# The data will be saved as a CSV file in the "data" folder. 

'''
import requests
import pandas as pd
from pathlib import Path

url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": 43.4643,
    "longitude": -80.5204,
    "start_date": "2025-01-01",
    "end_date": "2025-12-31",
    "hourly": [
        "temperature_2m",
        "relative_humidity_2m",
        "dew_point_2m",
        "apparent_temperature",
        "precipitation",
        "rain",
        "snowfall",
        "weather_code",
        "surface_pressure",
        "cloud_cover",
        "wind_speed_10m",
        "wind_direction_10m",
        "wind_gusts_10m"
    ],
    "timezone": "America/Toronto",
    "temperature_unit": "celsius",
    "wind_speed_unit": "kmh",
    "precipitation_unit": "mm"
}

print("Fetching data from Open-Meteo...")

response = requests.get(url, params=params, timeout=60)

response.raise_for_status()

data = response.json()

df = pd.DataFrame(data["hourly"])

df["latitude"] = data["latitude"]
df["longitude"] = data["longitude"]
df["elevation"] = data["elevation"]
df["timezone"] = data["timezone"]

df["time"] = pd.to_datetime(df["time"])


output_folder = Path("data")
output_folder.mkdir(parents=True, exist_ok=True)

output_file = output_folder / "waterloo_weather_2025.csv"

df.to_csv(output_file, index=False)


print("\nData downloaded successfully!")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print(f"Saved to: {output_file.resolve()}")

print("\nFirst five rows:")
print(df.head())

print("\nDataset information:")
df.info()

print("\nMissing values:")
print(df.isnull().sum())

'''

'\nimport requests\nimport pandas as pd\nfrom pathlib import Path\n\nurl = "https://archive-api.open-meteo.com/v1/archive"\n\nparams = {\n    "latitude": 43.4643,\n    "longitude": -80.5204,\n    "start_date": "2025-01-01",\n    "end_date": "2025-12-31",\n    "hourly": [\n        "temperature_2m",\n        "relative_humidity_2m",\n        "dew_point_2m",\n        "apparent_temperature",\n        "precipitation",\n        "rain",\n        "snowfall",\n        "weather_code",\n        "surface_pressure",\n        "cloud_cover",\n        "wind_speed_10m",\n        "wind_direction_10m",\n        "wind_gusts_10m"\n    ],\n    "timezone": "America/Toronto",\n    "temperature_unit": "celsius",\n    "wind_speed_unit": "kmh",\n    "precipitation_unit": "mm"\n}\n\nprint("Fetching data from Open-Meteo...")\n\nresponse = requests.get(url, params=params, timeout=60)\n\nresponse.raise_for_status()\n\ndata = response.json()\n\ndf = pd.DataFrame(data["hourly"])\n\ndf["latitude"] = data["latitude"]\ndf

In [2]:
import sys
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from dotenv import load_dotenv

# Set up project paths so the notebook can access our Python classes.
project_root = Path.cwd().resolve()

if not (project_root / "src").is_dir():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Load database settings and import the streaming classes.
load_dotenv(project_root / ".env")

from src.weather_stream import WeatherDatabase, WeatherStreamer

csv_path = project_root / "data" / "waterloo_weather_2025.csv"

print("Project root:", project_root)
print("CSV exists:", csv_path.exists())
print("Setup complete!")

Project root: C:\Users\KING\Desktop\KNearestNeighbors_Workshop
CSV exists: True
Setup complete!


In [3]:
# Load the CSV and check its structure and missing values.
df = pd.read_csv(csv_path)

print("Dataset shape:", df.shape)
display(df.head())

print("\nMissing values:")
display(df.isna().sum())

Dataset shape: (8760, 18)


,time,temperature_2m,relative_humidity_2m,dew_point_2m,apparent_temperature,precipitation,rain,snowfall,weather_code,surface_pressure,cloud_cover,wind_speed_10m,wind_direction_10m,wind_gusts_10m,latitude,longitude,elevation,timezone
0,2025-01-01 00:00:00,0.7,96,0.2,-3.8,1.0,0.1,0.63,73,962.1,100,16.3,6,32.4,43.479786,-80.54056,324.0,America/Toronto
1,2025-01-01 01:00:00,0.6,96,0.1,-4.1,1.0,0.0,0.70,73,962.2,100,17.6,354,32.4,43.479786,-80.54056,324.0,America/Toronto
2,2025-01-01 02:00:00,0.6,95,-0.1,-4.4,0.8,0.1,0.49,73,962.4,100,19.6,343,36.0,43.479786,-80.54056,324.0,America/Toronto
3,2025-01-01 03:00:00,0.6,94,-0.3,-4.7,0.7,0.1,0.42,73,962.7,100,20.9,339,38.9,43.479786,-80.54056,324.0,America/Toronto
4,2025-01-01 04:00:00,0.6,94,-0.4,-4.9,0.3,0.0,0.21,73,963.3,100,22.2,334,41.4,43.479786,-80.54056,324.0,America/Toronto



Missing values:


time                    0
temperature_2m          0
relative_humidity_2m    0
dew_point_2m            0
apparent_temperature    0
precipitation           0
rain                    0
snowfall                0
weather_code            0
surface_pressure        0
cloud_cover             0
wind_speed_10m          0
wind_direction_10m      0
wind_gusts_10m          0
latitude                0
longitude               0
elevation               0
timezone                0
dtype: int64

In [4]:
# Connect to Neon and display the latest saved weather records.
db = WeatherDatabase()

try:
    db.connect()

    print("Connected to Neon!")
    print("Rows currently stored:", db.count_rows())

    records = db.fetch_latest(limit=10)
    display(pd.DataFrame(records))

finally:
    db.close()

Connected to Neon!
Rows currently stored: 32


,id,time,temperature_2m,relative_humidity_2m,dew_point_2m,apparent_temperature,precipitation,rain,snowfall,weather_code,surface_pressure,cloud_cover,wind_speed_10m,wind_direction_10m,wind_gusts_10m,latitude,longitude,elevation,timezone,streamed_at
0,23,2025-01-01 22:00:00,-2.0,81.0,-4.9,-6.9,0.2,0.0,0.14,71,972.4,99.0,15.3,269.0,29.5,43.479786,-80.54056,324.0,America/Toronto,2026-10-09 15:56:11.851694
1,24,2025-01-01 23:00:00,-2.0,82.0,-4.7,-7.1,0.9,0.0,0.63,73,972.5,95.0,16.2,273.0,31.3,43.479786,-80.54056,324.0,America/Toronto,2026-10-09 15:56:11.982024
2,25,2025-01-02 00:00:00,-1.9,84.0,-4.2,-6.9,0.4,0.0,0.28,73,972.7,90.0,16.0,281.0,31.0,43.479786,-80.54056,324.0,America/Toronto,2026-10-09 15:56:14.084313
3,26,2025-01-02 01:00:00,-2.1,84.0,-4.4,-7.0,0.1,0.0,0.07,71,972.9,95.0,15.6,281.0,30.2,43.479786,-80.54056,324.0,America/Toronto,2026-10-09 15:56:16.188011
4,27,2025-01-02 02:00:00,-1.9,85.0,-4.1,-7.3,0.3,0.0,0.21,73,973.2,100.0,18.8,280.0,35.3,43.479786,-80.54056,324.0,America/Toronto,2026-10-09 15:56:18.293234
5,28,2025-01-02 03:00:00,-2.3,86.0,-4.3,-8.1,0.1,0.0,0.07,71,973.8,83.0,21.3,284.0,40.7,43.479786,-80.54056,324.0,America/Toronto,2026-10-09 15:56:20.406257
6,29,2025-01-02 04:00:00,-2.1,85.0,-4.3,-7.8,0.0,0.0,0.07,71,974.4,91.0,21.0,283.0,41.4,43.479786,-80.54056,324.0,America/Toronto,2026-10-09 15:56:22.514998
7,30,2025-01-02 05:00:00,-2.6,87.0,-4.5,-8.3,0.2,0.0,0.14,71,974.6,100.0,20.9,278.0,41.8,43.479786,-80.54056,324.0,America/Toronto,2026-10-09 15:56:24.620499
8,31,2025-01-02 06:00:00,-2.5,88.0,-4.2,-8.8,0.4,0.0,0.28,73,974.7,98.0,24.8,279.0,44.3,43.479786,-80.54056,324.0,America/Toronto,2026-10-09 15:56:26.730173
9,32,2025-01-02 07:00:00,-2.9,86.0,-4.9,-9.0,0.4,0.0,0.28,73,975.0,100.0,22.8,279.0,47.2,43.479786,-80.54056,324.0,America/Toronto,2026-10-09 15:56:28.841429


In [5]:
# Resume streaming from the latest saved timestamp.
db = WeatherDatabase()

try:
    db.create_table()
    resume_from = db.last_streamed_time()

    # Send one CSV record to Neon every two seconds.
    streamer = WeatherStreamer(
        csv_path=str(csv_path),
        interval=2
    )

    streamer.subscribe(db.insert_reading)

    print("Starting 5-record stream...")

    sent = streamer.run(
        start_after=resume_from,
        limit=5
    )

    print(f"Streamed {sent} records.")
    print("Total records in Neon:", db.count_rows())

finally:
    db.close()

Starting 5-record stream...
Streamed 5 records.
Total records in Neon: 37


In [ ]:
import time
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

# Create four interactive weather charts.
fig = go.FigureWidget(
    make_subplots(
        rows=2,
        cols=2,
        subplot_titles=(
            "Temperature (°C)",
            "Humidity (%)",
            "Wind Speed (km/h)",
            "Precipitation (mm)"
        )
    )
)

weather_columns = [
    "temperature_2m",
    "relative_humidity_2m",
    "wind_speed_10m",
    "precipitation"
]

for i, column in enumerate(weather_columns):
    fig.add_trace(
        go.Scatter(
            x=[],
            y=[],
            mode="lines+markers",
            name=column
        ),
        row=(i // 2) + 1,
        col=(i % 2) + 1
    )

fig.update_layout(
    title="Waterloo Weather — Live Streaming Dashboard",
    height=700,
    showlegend=False
)

display(fig)

# Load the latest saved records and update the charts.
def refresh_dashboard():
    records = db.fetch_latest(limit=100)
    data = pd.DataFrame(records)

    if data.empty:
        return

    with fig.batch_update():
        for i, column in enumerate(weather_columns):
            fig.data[i].x = data["time"].tolist()
            fig.data[i].y = data[column].tolist()


# Stream new CSV records into Neon and refresh after each insert.
db = WeatherDatabase()

try:
    db.connect()
    resume_from = db.last_streamed_time()

    streamer = WeatherStreamer(
        csv_path=str(csv_path),
        interval=2
    )

    streamer.subscribe(db.insert_reading)
    streamer.subscribe(lambda record: refresh_dashboard())

    refresh_dashboard()

    print("Streaming weather data and updating charts...")

    sent = streamer.run(
        start_after=resume_from,
        limit=10
    )

    print(f"Finished streaming {sent} records.")
    print("Total records in Neon:", db.count_rows())

finally:
    db.close()

FigureWidget({
    'data': [{'mode': 'lines+markers',
              'name': 'temperature_2m',
              'type': 'scatter',
              'uid': '9e5070fc-50d2-4a77-8a36-bd6d647f99fa',
              'x': [],
              'xaxis': 'x',
              'y': [],
              'yaxis': 'y'},
             {'mode': 'lines+markers',
              'name': 'relative_humidity_2m',
              'type': 'scatter',
              'uid': '8a773a3d-0b66-496d-8b4a-81c7274c01f7',
              'x': [],
              'xaxis': 'x2',
              'y': [],
              'yaxis': 'y2'},
             {'mode': 'lines+markers',
              'name': 'wind_speed_10m',
              'type': 'scatter',
              'uid': 'f86d52de-f0ff-4e42-87d1-13c47e04bf85',
              'x': [],
              'xaxis': 'x3',
              'y': [],
              'yaxis': 'y3'},
             {'mode': 'lines+markers',
              'name': 'precipitation',
              'type': 'scatter',
              'uid': '385b8631-b92f-

Streaming weather data and updating charts...
Finished streaming 10 records.
Total records in Neon: 47


## 2. Cleaning and preprocessing (Eche)
- Grouped the weather: Labelled each record as Clear, Cloudy, Rain, or Snow.
- Removed columns that could give the model the answer.
- Extracted the `hour` and `month`, handled missing values, and scaled the numbers.
- Used 80% for training and 20% for testing.

In [ ]:
class Preprocessor:
    LABEL_MAP = {
        **{code: "Clear" for code in range(0, 2)},
        **{code: "Cloudy" for code in range(2, 4)},
        **{code: "Rain" for code in list(range(51, 68)) + list(range(80, 83))},
        **{code: "Snow" for code in list(range(71, 78)) + list(range(85, 87))},
    }
    LEAKAGE_COLUMNS = ["weather_code", "rain", "snowfall", "precipitation"]
    CLASS_NAMES = ["Clear", "Cloudy", "Rain", "Snow"]

    def prepare_features_and_target(self, dataframe):
        data = dataframe.copy()
        print("Missing values before preprocessing:")
        display(data.isna().sum().to_frame("missing_count"))

        data["weather_code"] = pd.to_numeric(data["weather_code"], errors="coerce")
        data["condition"] = data["weather_code"].map(self.LABEL_MAP)

        # Parse timestamps and derive useful time-based predictors.
        timestamp = pd.to_datetime(data["time"], errors="coerce")
        data["hour"] = timestamp.dt.hour
        data["month"] = timestamp.dt.month

        # Drop rows where target/time cannot be determined.
        before = len(data)
        data = data.dropna(subset=["condition", "hour", "month"])
        print(f"Rows excluded because they do not map to a target class or valid time: {before-len(data)}")

        # Drop raw time, target source, and leakage variables from features.
        drop_cols = self.LEAKAGE_COLUMNS + ["condition", "time"]
        X = data.drop(columns=[c for c in drop_cols if c in data.columns])
        y = data["condition"]

        # Use numeric predictors only; non-numeric metadata such as timezone is not a model feature.
        X = X.select_dtypes(include=[np.number]).copy()
        print("\nMissing values in numeric features after target/time filtering:")
        display(X.isna().sum().to_frame("missing_count"))
        print("\nClass counts:")
        display(y.value_counts().reindex(self.CLASS_NAMES, fill_value=0).to_frame("count"))
        return X, y

preprocessor = Preprocessor()
X, y = preprocessor.prepare_features_and_target(df)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Fit imputer and scaler on training data only, then transform test data.
imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

print("Training rows:", len(X_train_scaled))
print("Test rows:", len(X_test_scaled))
print("Features:", list(X.columns))

Missing values before preprocessing:


,missing_count
time,0
temperature_2m,0
relative_humidity_2m,0
dew_point_2m,0
apparent_temperature,0
precipitation,0
rain,0
snowfall,0
weather_code,0
surface_pressure,0


Rows excluded because they do not map to a target class or valid time: 0

Missing values in numeric features after target/time filtering:


,missing_count
temperature_2m,0
relative_humidity_2m,0
dew_point_2m,0
apparent_temperature,0
surface_pressure,0
cloud_cover,0
wind_speed_10m,0
wind_direction_10m,0
wind_gusts_10m,0
latitude,0



Class counts:


,count
condition,
Clear,3079
Cloudy,3950
Rain,900
Snow,831


Training rows: 7008
Test rows: 1752
Features: ['temperature_2m', 'relative_humidity_2m', 'dew_point_2m', 'apparent_temperature', 'surface_pressure', 'cloud_cover', 'wind_speed_10m', 'wind_direction_10m', 'wind_gusts_10m', 'latitude', 'longitude', 'elevation', 'hour', 'month']
